# FireSat II - screening LCA with Brightway

**What you do in this notebook**
1. Get to know the LCA software "Brightway", which is a open source python package  (databases, datasets, life cycle impact assessment (LCIA calculation + visualisation).
2. Import the FireSat II life cycle inventory (LCI) from `data/FireSat2_LCI.xlsx` (your team edits this file).
3. Calculate impacts and break them down to single inventory lines.
4. Interpret: hotspots, data quality, flows without impacts.

**How to work:** run cells from top to bottom with Shift+Enter. You change the **Excel file**, not the code during this exercise. After each change,
re-run the code from section 3 onwards. Cells marked ✏️ are for your team.
The task for your team is in `EXERCISE.md`.

More information on Brightway and example for training materials: 
https://brightway.dev/

Many open source python packages have been built around Brightway to perform LCA: https://premisedash-6f5a0259c487.herokuapp.com/ecosystem/

Tutorial: Day 1 of https://github.com/romainsacchi/aalborg-regionalized-lcia 
This tutorial works with a free, open LCI database released last year by BAFU: https://nexus.openlca.org/database/BAFU 

User interfaces: https://github.com/LCA-ActivityBrowser/activity-browser or the recently released https://github.com/Phurga/lca_explorer 

In [ ]:
import bw2data as bd
import bw2calc as bc
import pandas as pd

#Normally, you would type some code to do the LCA with Brightway. For this exercise, all important functions are already prepared for you in the 
#lca_helpers.py file. We import this file and use the functions in it. If you are curious, have a look at the code in lca_helpers.py to see how the functions are implemented.
import lca_helpers as h

pd.set_option("display.max_colwidth", 80)

#Brightway works with projects. You can create them from scratch, or import a project from a .tar project backup file (which is what you did as preparation for this exercise)

bd.projects.set_current("eth_space_lca_2026")

#A newly created project does not have any databases - these need to be imported first. Usually, you would start with importing ecoinvent and the ESA LCA database. 
#For this exercise, the backup project already contains all necessary databases. Let's check if they are there. 
bd.databases

#You should see
# - ecoinvent-3.12-biosphere, which contains all elementary flows (matrix B = biosphere in the lecture material)
# - ecoinvent-3.12-cutoff, which contains all technosphere exchanges (matrix A = technosphere in the lecture material)
# - ESA_LCA_selection_e3.12 - a selection of datasets from the ESA database. These are also added to matrix A

## 1  First understanding of Brightway

A Brightway **project** holds **databases**. A database holds **activities** (or nodes when adopting the supply chain graph thinking) (= datasets, processes).
Each activity has **exchanges** (or "edges" in the supply chain graph thinking): inputs from other activities (technosphere) and emissions/resources (biosphere).


![title](../docs/bw_structure.png)


| Database | What it is |
|---|---|
| `ecoinvent-3.12-cutoff` | background: energy, materials, transport (licence of PSI - do not share) |
| `ESA_LCA_selection_e3.12` | space-specific datasets from the ESA LCA database 1.3.0b (selection) |
| `ecoinvent-3.12-biosphere` | elementary flows: emissions and resources |
| `space_emissions` | two flows ecoinvent does not have (Al₂O₃ and HCl in the stratosphere) |
| `firesat2` | **your foreground**, imported from the Excel |

Search a database (the same as the search field in Activity Browser):

In [ ]:
h.search("solar", h.ESA)

Look into one dataset - what goes in?

In [ ]:
solar = h.get_activity("solar array production, power supply", h.ESA)
print(solar["name"], "|", solar["unit"], "|", solar["location"])
print(solar.get("comment", "")[:400], "...\n")
for exc in solar.technosphere():
    print(f"{exc['amount']:>10.4g} {exc.input['unit']:15} {exc.input['name'][:70]}")

**Impact assessment methods** are stored per project. We use the EF v3.1 method (the method of the EU
Environmental Footprint, also used in the ESA LCA Handbook). The ESA LCA Handbook (Annex B) asks at least for
climate change, human toxicity and resource use (minerals and metals), plus ozone depletion when the launch is
inside the system boundary.

In [ ]:
EF = ("ecoinvent-3.12", "EF v3.1")
METHODS = [
    EF + ("climate change", "global warming potential (GWP100)"),
    EF + ("ozone depletion", "ozone depletion potential (ODP)"),
    EF + ("material resources: metals/minerals", "abiotic depletion potential (ADP): elements (ultimate reserves)"),
    EF + ("human toxicity: carcinogenic", "comparative toxic unit for human (CTUh)"),
    EF + ("human toxicity: non-carcinogenic", "comparative toxic unit for human (CTUh)"),
]
GWP = METHODS[0]
[m for m in bd.methods if m[:2] == EF][:8]

**First LCIA calculation.** This is the calculation from the lecture: Brightway builds the technosphere matrix **A**,
the biosphere matrix **B** and the characterisation matrix **C**, solves **A s = f** and computes **h = C B s**.

In [ ]:
fu, data_objs, _ = bd.prepare_lca_inputs({solar: 1}, method=GWP)
lca = bc.LCA(fu, data_objs=data_objs)
lca.lci()     # solve A s = f, then g = B s (inventory)
lca.lcia()    # h = C g
print("A:", lca.technosphere_matrix.shape, " B:", lca.biosphere_matrix.shape)
print(f"1 kg solar array: {lca.score:.1f} kg CO2-eq")

## 2  Goal and scope ✏️

Fill in the sheet **goal_scope** of `data/FireSat2_LCI.xlsx` (you may use your morning work).
The draft functional unit in the workbook is *one FireSat II mission (2 spacecraft, 8 years) in fulfilment of
its requirements* (ESA LCA Handbook 6.2.1.1). Keep it unless your team has a good reason to change it; and think of an appropriate functional unit for your work package dataset.

In [ ]:
#You can display what you wrote directly here in the notebook if you want. 

XLSX = "../data/FireSat2_LCI.xlsx"
pd.read_excel(XLSX, sheet_name="goal_scope", header=None, names=["item", "your text from the excel"]).fillna("")

## 3  Life cycle inventory: import the foreground ✏️

The sheet **lci** holds the foreground. Each activity block lists its exchanges with a **data quality rating**
(DQR, 1 = rough ... 4 = very good, see sheet *dqr_guide*), a **source** and a **comment**.

After every change in the Excel: **save the file** and re-run from here. If an exchange cannot be linked
(typo in name, location or unit), you get a table telling you which one.

In [ ]:
#we first import the foreground data from the Excel file. This will create a new database in the current project (or overwrite the existing one)

h.import_foreground(XLSX)

#Let's check which activities we have imported by looping through all reference products in our database 
#after you have re-imported your excel sheet, you can check if all activities are there (including new ones if you created any)
FU = h.get_activity("FireSat II mission, screening LCA")
for act in bd.Database(h.FG):
    print(f"{act['name']:65} {len(list(act.exchanges())) - 1:>3} exchanges")

Brightway now knows the whole supply chain. `exchange_table` lists every line of the Excel that goes into the
background or the biosphere, with the amount needed **per functional unit** (`per_fu`):

In [ ]:
inventory = h.exchange_table(FU)
inventory[["stage", "parent", "input", "per_fu", "unit", "dqr"]].head(15)

## 4  Impact assessment

This part calculates all impact categories, and the impact of 1 unit of every input, in one go by using a multi-LCA function
(`bc.MultiLCA`, one matrix factorisation). `check()` confirms that the single contributions add up to the total.

### Mission or work package? ✏️

By default you analyse the **whole mission**. Your work package may be only a few percent of the mission, so its hotspots are hard to see in the plots. To look at your work package in detail:

1. In the next cell, set `SYSTEM` to your team letter, e.g. `SYSTEM = "B"`.
2. Run sections 4 and 5 again: all plots now show your work package only.
3. For the mission result on your slide, set `SYSTEM = "mission"` again and re-run sections 4 and 5.

`TEAMS` lists which activities of the sheet `lci` belong to each team, and how much of each one mission needs (the reference flows, e.g. 2 spacecraft). So your results are always **your part of one mission**, and the totals of A + B + C + D add up to the mission. Shares (%) refer to your part, not to the mission.

**If you add a new activity** in the Excel (e.g. team D: end of life):

1. In the Excel, add it as an input of `FireSat II mission, screening LCA` (amount: how much of it one mission needs, usually 1). Otherwise it is missing in the mission result.
2. In `TEAMS`, add it to your team's line in the format `"activity name": amount`, separated by a comma from the other activities, e.g.
   `"D": {"FireSat II ground segment and operations, 8 years": 1, "FireSat II end of life": 1},`
   The name must be exactly the one in the Excel (upper/lower case, spaces).

You can also analyse any single activity, e.g. `res = h.analyse({"TVAC test day, small chamber": 1}, METHODS)`.

In [ ]:
# ✏️ Which system do you analyse? "mission" (default) or your team: "A", "B", "C" or "D"
SYSTEM = "mission"

# Format of each line:  "team": {"activity name exactly as in the Excel": how much of it one mission needs,
#                                "next activity name": amount},
TEAMS = {
    "mission": {"FireSat II mission, screening LCA": 1},
    "A": {"FireSat II spacecraft, flight model": 2},
    "B": {"FireSat II AIT and test campaign": 1,
          "FireSat II engineering, office work and travel": 1},
    "C": {"FireSat II launch segment, 2 x Minotaur IV": 1},
    "D": {"FireSat II ground segment and operations, 8 years": 1,
          # new activity: remove the # in the next line and write its exact name
          # "FireSat II end of life": 1,
          },
}

res = h.analyse(TEAMS[SYSTEM], METHODS)
print("Results for:", SYSTEM, TEAMS[SYSTEM])
res.summary()

In [ ]:
res.check()

## 5  Interpretation

### 5a  Which life cycle stage matters, and does it differ between impact categories?

In [ ]:
h.plot_stages(res)

### 5b  Hotspots and how good the data behind them are
Bars = share of the total. Colour = mean DQR of that inventory line (red = rough guess, green = very good data).
**A red bar at the top means your conclusion rests on a rough guess.**

In [ ]:
h.plot_dqr_hotspots(res, GWP)
h.top_contributors(res, GWP, n=10)

In [ ]:
h.plot_dqr_hotspots(res, METHODS[2])   # resource use - is the picture the same as for climate change?

### 5c  Where should you collect better data first?
x-axis = share of the total, y-axis = data quality. **Bottom right** (red area) = large contribution **and** low data quality. That is where better data could change the conclusion.

In [ ]:
h.plot_dqr_priority(res, GWP)

### 5d  Data quality of the whole study
Contribution-weighted DQR per impact category and per stage - the logic of the ESA LCA Handbook, Annex B
(there on a 1-5 scale where 1 is best; here 1-4 where 4 is best). Unrated lines count as 1.

In [ ]:
h.dqr_summary(res)

### 5e  Flows without impacts
The launch exhaust is in the inventory. Which of these flows get a characterisation factor (CF)?

In [ ]:
h.flows_without_cf(res)

**Discuss:** Al₂O₃ and HCl get a CF of 0 in every category - they simply do not count. What does that mean for
the "launch" bars in 5a? Which numbers would you report next to the LCA results (ESA LCA Handbook: launch
emissions are reported as elementary flows)?

### 5f  What if? Ecodesign and sensitivity
`what_if` changes single inventory lines in the calculation only (the database is not modified).
Three examples - then add your own ✏️:

*The three examples change lines of the whole mission: run them with `SYSTEM = "mission"`. With your work package as `SYSTEM`, write variants for lines inside it (a warning tells you when a change matched no line).*

In [ ]:
# 1) Rideshare on Falcon 9 instead of two dedicated Minotaur IV launches
rideshare = h.get_activity("VARIANT: rideshare launch on Falcon 9, mass-allocated share")
h.what_if(res,
          changes=[{"where": lambda r: r.stage.startswith("FireSat II launch segment"), "factor": 0}],
          add=[(rideshare, 1)],
          name="rideshare")

In [ ]:
# 2) Structure in CFRP-faced aluminium honeycomb, assumed 30 % lighter
honeycomb = h.get_activity("AL 5056 Honeycomb with CFRP facing [mass] production", h.ESA)
h.what_if(res,
          changes=[{"where": lambda r: r.input == "aluminium part for structure AA 7075" and r.amount == 23,
                    "new": honeycomb, "factor": 0.7}],
          name="CFRP honeycomb structure")

In [ ]:
# 3) "Green" propellant LMP-103S instead of hydrazine (same mass - is that right?)
lmp = h.get_activity("LMP-103S, spacecraft system propellant, propellant loading", h.ESA)
h.what_if(res,
          changes=[{"where": lambda r: r.input == "hydrazine, monopropellant, propellant use", "new": lmp}],
          name="LMP-103S")

In [ ]:
# ✏️ your own variant

### 5g  Export for your slides / report

In [ ]:
# Exports the current SYSTEM. Use another file name for your work package,
# e.g. "../results_team_B.xlsx", so you do not overwrite the mission results.
h.export(res, "../results_firesat2.xlsx")

## 6  Optional challenge: is this dataset right?

The ESA dataset `iSO8 Clean Room, Testing` has the unit *hour*. Its documentation says: 250 m² cleanroom,
**per day** 2880 kWh electricity and 400 m³ natural gas. Calculate 1 hour with Brightway and compare with what
you expect from the documentation. What went wrong, and what does this teach you about using databases?

In [ ]:
cleanroom = h.get_activity("iSO8 Clean Room, Testing", h.ESA)
print(cleanroom.get("comment", "")[:300])
for exc in cleanroom.exchanges():
    print(exc["type"], exc["amount"], exc.input["unit"], exc.input["name"])
# ✏️ your calculation